# EQ‑Proof (interactive notebook)

This notebook is an **app-like interface** for EQ‑Proof, built primarily as a **personal tool**.  
You are welcome to read, run, and adapt it — it is provided **as-is**.

## What it does
- Load a **spec** (constraints + units) and **inputs**
- Optionally **coerce units** into the spec’s canonical units
- **Diagnose** violations and **repair** inputs to satisfy constraints
- Generate a **tamper-evident proof report** (and optionally a PDF)
- **Verify** the report’s attestation

## How to use
1. Run the cells top-to-bottom once
2. Use the UI tabs (requires `ipywidgets`) or run the fallback demo

> Tip: This notebook is intentionally offline-first.  
> If you want network access for any reason, set `EQPROOF_ALLOW_NET=1` in your environment **before** running the notebook.


In [13]:
# config / utils
# This cell is deliberately "boring": it defines the tiny plumbing we reuse everywhere else.

from __future__ import annotations

import os
import time
import uuid
import platform
import json
import hashlib
from typing import Dict, Any, Optional


class Config:
    """Central place for tunables used by the repair algorithms.

    Notes:
    - The tolerances are intentionally small; if you work with noisy real-world data,
      you may want to increase them.
    - ALTPROJ_* settings are used by the 'altproj' (alternating projections) strategy.
    """

    # Equality constraints are considered satisfied when residual <= EQUALITY_TOL.
    EQUALITY_TOL: float = 1e-9

    # Alternating projections settings (used for 'altproj' strategy).
    ALTPROJ_ITERS: int = 200
    ALTPROJ_TOL: float = 1e-9  # stop when changes + residuals are below this

    # Notebook-friendly demo secret for HMAC signing if EQPROOF_KEY isn't provided.
    DEMO_KEY: bytes = b"DEMO_KEY"

    # If the Spec doesn't declare units, we use this empty default.
    DEFAULT_UNITS: Dict[str, str] = {}


def now_ms() -> int:
    """Epoch milliseconds."""
    return int(time.time() * 1000)


def run_meta() -> Dict[str, Any]:
    """Metadata captured in proof reports to help with audit/traceability."""
    return {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "time_ms": now_ms(),
        "run_id": str(uuid.uuid4()),
    }


def sha256_bytes(b: bytes) -> str:
    """Hex digest of SHA-256."""
    return hashlib.sha256(b).hexdigest()


def canonical_json(obj: Any) -> bytes:
    """Stable JSON encoding used for signing/verification."""
    return json.dumps(obj, sort_keys=True, separators=(",", ":")).encode("utf-8")


def log_step(report: Dict[str, Any], step: Dict[str, Any]) -> None:
    """Append an operation step to the report."""
    report.setdefault("steps", []).append(step)


def badge(ok: bool, text: str) -> str:
    """Small helper for UI messages."""
    return f"✅ {text}" if ok else f"❌ {text}"


In [14]:
# no_net (offline-first safety guard)
#
# EQ‑Proof is designed to be usable in sensitive/offline environments.
# By default we block outbound network connections at the socket layer.
#
# To allow outbound network (not recommended), set:
#   EQPROOF_ALLOW_NET=1
# before running this notebook.

import socket


class _NoNetSocket(socket.socket):
    def connect(self, *a, **k):
        raise RuntimeError("Outbound network disabled by eq_proof.no_net")

    def connect_ex(self, *a, **k):
        raise RuntimeError("Outbound network disabled by eq_proof.no_net")


def enforce_offline() -> None:
    # Only patch sockets if the user did not explicitly opt-in to networking.
    if os.environ.get("EQPROOF_ALLOW_NET", "0") not in ("1", "true", "True"):
        socket.socket = _NoNetSocket  # type: ignore[misc]


enforce_offline()


In [15]:
# spec
from dataclasses import dataclass, field
import json
from typing import Dict, Any, List

@dataclass
class Spec:
    name: str
    version: str
    variables: List[str]
    constraints: List[Dict[str, Any]]
    probes: List[Dict[str, Any]] = field(default_factory=list)
    alternates: List[str] = field(default_factory=list)
    units: Dict[str, str] = field(default_factory=dict)

def load_spec(path: str) -> Spec:
    with open(path, "r") as f: d = json.load(f)
    for k in ["name","version","variables","constraints"]:
        if k not in d: raise ValueError(f"Spec missing required field: {k}")
    return Spec(d["name"], d["version"], d["variables"], d["constraints"], d.get("probes",[]), d.get("alternates",[]), d.get("units",{}))

def spec_hash(spec: Spec) -> str: return sha256_bytes(canonical_json(spec.__dict__))


In [16]:
# units (minimal, intentionally small)
#
# EQ‑Proof supports optional unit coercion of inputs into the spec’s canonical units.
# This is a deliberately tiny unit system (enough for demos and many constraint checks).
#
# Supported base tokens: m, kg, s (plus dimensionless '')
# Supported derived tokens: Hz
#
# Extend BASE/DER if you need more.

from typing import Tuple, Dict

BASE = {
    "": ((0, 0, 0, 0, 0, 0, 0), 1.0),      # dimensionless
    "m": ((1, 0, 0, 0, 0, 0, 0), 1.0),     # meters
    "kg": ((0, 1, 0, 0, 0, 0, 0), 1.0),    # kilograms
    "s": ((0, 0, 1, 0, 0, 0, 0), 1.0),     # seconds
}
DER = {
    "Hz": ((0, 0, -1, 0, 0, 0, 0), 1.0),   # 1/s
}


def _mul(a, b):
    return tuple(x + y for x, y in zip(a, b))


def _div(a, b):
    return tuple(x - y for x, y in zip(a, b))


def _pow(a, p: int):
    return tuple(x * p for x in a)


def _atom(tok: str):
    if tok in BASE:
        return BASE[tok]
    if tok in DER:
        return DER[tok]
    raise ValueError(f"Unknown unit token: {tok}")


def parse_unit(u: str) -> Tuple[Tuple[int, ...], float]:
    """Parse a unit string like 'm', 'm/s', 'kg*m/s^2', 'Hz' into (dimension, factor)."""
    if not u or u == "1":
        return BASE[""]

    u = u.replace(" ", "")
    num, *den = u.split("/")
    dim = BASE[""][0]
    fac = 1.0

    for tok in filter(None, num.split("*")):
        base, p = tok.split("^") if "^" in tok else (tok, "1")
        p = int(p)
        d, f = _atom(base)
        dim = _mul(dim, _pow(d, p))
        fac *= f**p

    if den:
        den = "*".join(den)
        for tok in filter(None, den.split("*")):
            base, p = tok.split("^") if "^" in tok else (tok, "1")
            p = int(p)
            d, f = _atom(base)
            dim = _div(dim, _pow(d, p))
            fac /= f**p

    return dim, fac


def convert(val: float, from_u: str, to_u: str) -> float:
    """Convert numeric value from one unit to another (same physical dimension)."""
    d1, f1 = parse_unit(from_u)
    d2, f2 = parse_unit(to_u)
    if d1 != d2:
        raise ValueError("Incompatible units")
    return (val * f1) / f2


def coerce_inputs_to_spec_units(values: dict, spec_units: Dict[str, str]):
    """If an input is provided as {'value': X, 'unit': '...'}, convert into spec unit."""
    steps = []
    out = dict(values)
    for k, u in spec_units.items():
        if isinstance(values.get(k), dict) and "value" in values[k] and "unit" in values[k]:
            v = float(values[k]["value"])
            from_u = str(values[k]["unit"])
            v2 = convert(v, from_u, u)
            out[k] = v2
            steps.append(
                {"op": "unit_convert", "var": k, "from": from_u, "to": u, "value_in": v, "value_out": v2}
            )
    return out, steps


In [17]:
# constraints & repair
import sympy as sp
def equality_residual(expr_str: str, values: Dict[str, float]) -> float:
    locals_ = {**{k: sp.symbols(k, real=True) for k in values}, "Eq": sp.Eq}
    eq = sp.sympify(expr_str, locals=locals_)
    res = sp.simplify(eq.lhs - eq.rhs).subs({sp.Symbol(k): float(v) for k,v in values.items()})
    return float(sp.N(res))
def equality_solve_for(expr_str: str, target: str, values: Dict[str, float]) -> Optional[float]:
    syms = {k: sp.symbols(k, real=True) for k in set(values)|{target}}
    eq = sp.sympify(expr_str, locals={"Eq": sp.Eq, **syms})
    sol = sp.solve(eq, syms[target], dict=True)
    if not sol: return None
    return float(sp.N(sol[0][syms[target]].subs({syms[k]: float(v) for k,v in values.items()})))
def clip_bounds(vals: Dict[str, float], b: Dict[str, Tuple[Optional[float], Optional[float]]]):
    out = dict(vals)
    for k,(lo,hi) in b.items():
        v = out.get(k,0.0)
        if lo is not None: v = max(lo, v)
        if hi is not None: v = min(hi, v)
        out[k]=v
    return out


In [18]:
# diagnose
#
# The main entrypoint is diagnose_and_repair().
# It:
# 1) coerces units (optional),
# 2) applies one of the strategies,
# 3) returns original + repaired values plus a structured report.

def diagnose_and_repair(
    spec: Spec,
    values: Dict[str, float],
    strategy: str = "bounds_then_equality",
    *,
    spec_path: str = "",
    inputs_path: str = "",
    verbose: bool = True,
) -> Dict[str, Any]:
    started_ms = now_ms()

    report: Dict[str, Any] = {
        "violations": [],
        "steps": [],
        "meta": {"env": run_meta(), "strategy": strategy},
    }

    # 0) Unit coercion (only if spec declares units AND inputs provide unit objects).
    coerced, unit_steps = coerce_inputs_to_spec_units(values, getattr(spec, "units", Config.DEFAULT_UNITS))
    report["steps"].extend(unit_steps)

    original = dict(coerced)
    repaired = dict(coerced)

    # Pre-process constraints into a convenient form.
    bounds: Dict[str, Tuple[Optional[float], Optional[float]]] = {}
    equalities: List[Dict[str, Any]] = []
    for c in spec.constraints:
        if c.get("type") == "bounds":
            bounds[c["var"]] = (c.get("lower"), c.get("upper"))
        elif c.get("type") == "equality":
            equalities.append(c)

    def apply_bounds(vals: Dict[str, float]) -> Dict[str, float]:
        if not bounds:
            return vals
        before = {k: vals.get(k) for k in bounds.keys()}
        out = clip_bounds(vals, bounds)
        after = {k: out.get(k) for k in bounds.keys()}
        if before != after:
            log_step(report, {"op": "bounds_clip", "before": before, "after": after})
        return out

    def apply_equalities(vals: Dict[str, float]) -> Dict[str, float]:
        if not equalities:
            return vals
        out = dict(vals)
        for c in equalities:
            expr = c["expr"]
            tol = float(c.get("tol", Config.EQUALITY_TOL))
            res = abs(equality_residual(expr, out))
            if res > tol:
                target = c.get("solve_for")
                if target:
                    new = equality_solve_for(expr, target, out)
                    if new is not None:
                        before = out.get(target)
                        out[target] = float(new)
                        log_step(
                            report,
                            {
                                "op": "equality_solve",
                                "expr": expr,
                                "target": target,
                                "before": before,
                                "after": float(new),
                                "residual": res,
                            },
                        )
                else:
                    # We can detect the violation, but without a solve_for we won't mutate.
                    report["violations"].append({"type": "equality", "expr": expr, "residual": res})
        return out

    # Strategy behavior:
    # - bounds_then_equality: clip once, then solve equalities once.
    # - equality_only: solve equalities only (no bounds clipping).
    # - altproj: alternating projections (clip bounds <-> solve equalities) until stable.
    if strategy == "equality_only":
        repaired = apply_equalities(repaired)

    elif strategy == "bounds_then_equality":
        repaired = apply_bounds(repaired)
        repaired = apply_equalities(repaired)

    elif strategy == "altproj":
        # Alternating projections: enforce bounds and equalities repeatedly.
        # This is useful when solving equalities can push values out of bounds (and vice versa).
        iters = 0
        for i in range(int(Config.ALTPROJ_ITERS)):
            iters = i + 1
            prev = dict(repaired)

            repaired = apply_bounds(repaired)
            repaired = apply_equalities(repaired)

            # Evaluate convergence: max change + max equality residual
            max_delta = max((abs(repaired.get(k, 0.0) - prev.get(k, 0.0)) for k in set(prev) | set(repaired)), default=0.0)

            max_res = 0.0
            for c in equalities:
                try:
                    max_res = max(max_res, abs(equality_residual(c["expr"], repaired)))
                except Exception:
                    # If residual can't be computed, leave it for the report/violations.
                    pass

            if max_delta <= Config.ALTPROJ_TOL and max_res <= Config.ALTPROJ_TOL:
                break

        log_step(report, {"op": "altproj_summary", "iters": iters, "tol": Config.ALTPROJ_TOL})

    else:
        raise ValueError(f"Unknown strategy: {strategy}")

    report["meta"]["elapsed_ms"] = now_ms() - started_ms
    report["meta"]["spec_path"] = spec_path
    report["meta"]["inputs_path"] = inputs_path

    return {"original": original, "repaired": repaired, "report": report}


# Notebook session state used by the UI tabs below.
last_spec: Optional[Spec] = None
last_inputs: Optional[Dict[str, Any]] = None
last_result: Optional[Dict[str, Any]] = None
last_attestation: Optional[Dict[str, Any]] = None


In [19]:
# attest / verify
import hmac, hashlib, time
def _load_secret() -> bytes:
    key = os.environ.get("EQPROOF_KEY")
    if key: return key.encode("utf-8")
    print("[WARN] Using DEMO_KEY. Set EQPROOF_KEY for production."); return Config.DEMO_KEY
def attest(spec: dict, proof: dict, *, spec_path: str = "", inputs_path: str = "") -> dict:
    payload = {"spec": spec, "proof": proof, "meta": {"spec_hash": sha256_bytes(canonical_json(spec)), "inputs_hash": sha256_bytes(canonical_json(proof.get("original", {}))), "engine_version": "0.3.0", "runtime_env": run_meta(), "spec_path": spec_path, "inputs_path": inputs_path}, "ts": int(time.time())}
    msg = canonical_json(payload); sig = hmac.new(_load_secret(), msg, hashlib.sha256).hexdigest()
    payload["signature"] = sig; payload["algo"] = "HMAC-SHA256"; return payload
def _payload(att: dict) -> bytes:
    core = {k:v for k,v in att.items() if k not in ("signature","algo","pubkey")}; return canonical_json(core)
def verify_hmac(att: dict, key: str = "DEMO_KEY") -> bool:
    msg=_payload(att); calc=hmac.new(key.encode("utf-8"), msg, hashlib.sha256).hexdigest(); return calc == att.get("signature")


In [20]:
# report / pdf
import datetime
try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None
def render_markdown(spec_path: str, inputs_path: str, result: dict, attestation: dict, verbose: bool = True) -> str:
    ts = datetime.datetime.utcnow().isoformat()+"Z"; o=result["original"]; r=result["repaired"]; steps=result["report"].get("steps", [])
    rows = "\n".join([f"| {k} | {o.get(k)} | {r.get(k)} |" for k in sorted(set(o)|set(r))])
    md = [f"# EQ‑PROOF Report", f"- Generated: {ts}", f"- Spec: `{spec_path}`", f"- Inputs: `{inputs_path}`", "\n## Original vs Repaired", "| Variable | Original | Repaired |", "|---|---:|---:|", rows]
    if verbose:
        md.append("\n## Steps"); md += (["- "+json.dumps(s) for s in steps] or ["- none"]); md.append("\n## Attestation"); md.append(f"- algorithm: {attestation.get('algo')}"); md.append(f"- signature: `{attestation.get('signature')}`")
    return "\n".join(md)
def report_lines(spec_path: str, inputs_path: str, result: dict, attestation: dict) -> List[str]:
    lines=[f"Spec: {spec_path}", f"Inputs: {inputs_path}"]; o=result["original"]; r=result["repaired"]; steps=result["report"].get("steps", [])
    lines.append(""); lines.append("Original vs Repaired");
    for k in sorted(set(o)|set(r)): lines.append(f"- {k}: {o.get(k)} -> {r.get(k)}")
    lines.append(""); lines.append("Steps:"); lines += [json.dumps(s) for s in steps]; lines.append(""); lines.append(f"Attestation: {attestation.get('algo')} {attestation.get('signature')}"); return lines
def save_text_pdf(lines: List[str], out_path: str, title: str = "EQ‑PROOF Report", title_size: int = 16, body_size: int = 9) -> None:
    if plt is None: raise RuntimeError("matplotlib not available for PDF export")
    fig = plt.figure(figsize=(8.27, 11.69)); ax = fig.add_axes([0,0,1,1]); ax.axis('off')
    ax.text(0.05, 0.95, title, va='top', ha='left', fontsize=title_size, family='monospace')
    ax.text(0.05, 0.90, "\n".join(lines), va='top', ha='left', fontsize=body_size, family='monospace')
    fig.savefig(out_path, format='pdf', bbox_inches='tight'); plt.close(fig)


In [21]:
# interactive UI (tabs)
try:
    import ipywidgets as widgets
    from IPython.display import display, Markdown
    UI_AVAILABLE = True
except Exception:
    UI_AVAILABLE = False
if not UI_AVAILABLE:
    print("ipywidgets not available. Install with 'pip install ipywidgets' and enable it to use the interactive UI.")
else:
    # File browse dialogs (spec + inputs)
    spec_upload = widgets.FileUpload(accept='.json', multiple=False, description='Upload spec')
    inputs_upload = widgets.FileUpload(accept='.json', multiple=False, description='Upload inputs')
    # Text input fields
    inputs_area = widgets.Textarea(value='{"x": 12, "y": 3, "cap": 14}', description='Inputs JSON', layout=widgets.Layout(width='100%', height='150px'))
    att_text = widgets.Textarea(value='{"algo":"HMAC-SHA256","signature":"..."}', description='Attestation JSON', layout=widgets.Layout(width='100%', height='120px'))
    # Multiple choice selections
    strategy_dd = widgets.Dropdown(options=[('Bounds then equality','bounds_then_equality'),('Equality only','equality_only'),('Alternating projections','altproj')], value='bounds_then_equality', description='Strategy')
    verbose_toggle = widgets.ToggleButtons(options=[('Detailed','detailed'),('Short','short')], value='detailed', description='Report')
    # Buttons and outputs
    run_btn = widgets.Button(description='Run EQ‑Proof', button_style='primary')
    viz_btn = widgets.Button(description='Show chart')
    verify_btn = widgets.Button(description='Verify attestation')
    export_md_btn = widgets.Button(description='Download Markdown')
    export_pdf_btn = widgets.Button(description='Download PDF')
    status_out = widgets.Output(); results_out = widgets.Output(); viz_out = widgets.Output(); verify_out = widgets.Output(); export_out = widgets.Output()

    def parse_spec_upload() -> Optional[Spec]:
        if not spec_upload.value:
            return None
        try:
            content = list(spec_upload.value.values())[0]['content']
            d = json.loads(content.decode())
            return Spec(d['name'], d['version'], d['variables'], d['constraints'], d.get('probes',[]), d.get('alternates',[]), d.get('units',{}))
        except Exception as e:
            with status_out: status_out.clear_output(); print(badge(False, f"Spec upload invalid: {e}"))
            return None

    def parse_inputs_source() -> Dict[str, Any]:
        if inputs_upload.value:
            try:
                content = list(inputs_upload.value.values())[0]['content']
                return json.loads(content.decode())
            except Exception as e:
                with status_out: status_out.clear_output(); print(badge(False, f"Inputs upload invalid: {e}"))
        try:
            return json.loads(inputs_area.value)
        except Exception as e:
            with status_out: status_out.clear_output(); print(badge(False, f"Inputs JSON invalid: {e}"))
            return {}

    def on_run_clicked(_):
        results_out.clear_output(); viz_out.clear_output(); status_out.clear_output()
        spec = parse_spec_upload() or Spec(
            name="DemoSpec", version="0.3", variables=["x","y","cap"],
            constraints=[{"type":"bounds","var":"x","lower":0,"upper":10},{"type":"equality","expr":"Eq(x+y,cap)","solve_for":"y"}],
            units={}
        )
        inputs = parse_inputs_source()
        if not inputs:
            with status_out: print("Provide valid inputs and try again."); return
        global last_spec, last_inputs, last_result, last_attestation
        last_spec, last_inputs = spec, inputs
        result = diagnose_and_repair(spec, inputs, strategy=strategy_dd.value, spec_path='uploaded_or_demo.json', inputs_path='inline_or_uploaded_inputs.json', verbose=True)
        last_result = result
        att = attest(spec.__dict__, result, spec_path='uploaded_or_demo.json', inputs_path='inline_or_uploaded_inputs.json')
        last_attestation = att
        md = render_markdown('uploaded_or_demo.json', 'inline_or_uploaded_inputs.json', result, att, verbose=(verbose_toggle.value=='detailed'))
        with results_out: results_out.clear_output(); display(Markdown(md))
        with status_out: print(badge(True, "Run complete"))

    run_btn.on_click(on_run_clicked)

    def plot_before_after(result: dict):
        if plt is None:
            print("matplotlib unavailable: charts skipped"); return
        o = result["original"]; r = result["repaired"]
        labels = list(sorted(set(o)|set(r)))
        orig_vals = [o.get(k,0) for k in labels]
        rep_vals = [r.get(k,0) for k in labels]
        import numpy as np
        x = np.arange(len(labels))
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(8,4))
        ax.bar(x-0.2, orig_vals, width=0.4, label="Original")
        ax.bar(x+0.2, rep_vals, width=0.4, label="Repaired")
        ax.set_xticks(x); ax.set_xticklabels(labels)
        ax.legend(); ax.set_ylabel("Value"); ax.set_title("Original vs Repaired")
        plt.show()

    def on_viz_clicked(_):
        viz_out.clear_output()
        with viz_out:
            if last_result is None:
                print("Run the pipeline first.")
            else:
                plot_before_after(last_result)

    viz_btn.on_click(on_viz_clicked)

    def on_verify_clicked(_):
        verify_out.clear_output()
        try:
            att_obj = json.loads(att_text.value)
            ok = verify_hmac(att_obj, key=os.environ.get('EQPROOF_KEY','DEMO_KEY'))
            with verify_out: print(badge(ok, "Attestation verified" if ok else "Attestation invalid"))
        except Exception as e:
            with verify_out: print(badge(False, f"Verification failed: {e}"))

    verify_btn.on_click(on_verify_clicked)

    def on_export_md(_):
        export_out.clear_output()
        if last_result is None or last_attestation is None:
            with export_out: print("Run the pipeline first."); return
        md = render_markdown('uploaded_or_demo.json', 'inline_or_uploaded_inputs.json', last_result, last_attestation, verbose=True)
        path = 'eqproof_report.md'
        with open(path, 'w') as f: f.write(md)
        with export_out: print(badge(True, f"Saved {path}"))

    export_md_btn.on_click(on_export_md)

    def on_export_pdf(_):
        export_out.clear_output()
        if last_result is None or last_attestation is None:
            with export_out: print("Run the pipeline first."); return
        lines = report_lines('uploaded_or_demo.json','inline_or_uploaded_inputs.json', last_result, last_attestation)
        try:
            save_text_pdf(lines, 'eqproof_report.pdf')
            with export_out: print(badge(True, "Saved eqproof_report.pdf"))
        except Exception as e:
            with export_out: print(badge(False, f"PDF export failed: {e}"))

    export_pdf_btn.on_click(on_export_pdf)

    tabs = widgets.Tab(children=[
        widgets.VBox([widgets.HTML("<b>Setup:</b> Upload spec or inputs, or use demo."), widgets.HBox([spec_upload, inputs_upload]), inputs_area, widgets.HBox([strategy_dd, verbose_toggle]), status_out]),
        widgets.VBox([widgets.HTML("<b>Run:</b> Execute EQ‑Proof."), run_btn, status_out]),
        widgets.VBox([widgets.HTML("<b>Results:</b> Report and visualization."), results_out, widgets.HBox([viz_btn]), viz_out]),
        widgets.VBox([widgets.HTML("<b>Verify:</b> Paste attestation JSON."), att_text, verify_btn, verify_out]),
        widgets.VBox([widgets.HTML("<b>Export:</b> Save Markdown or PDF."), widgets.HBox([export_md_btn, export_pdf_btn]), export_out])
    ])
    for i,t in enumerate(["Setup","Run","Results","Verify","Export"]): tabs.set_title(i, t)
    display(tabs)


In [22]:
# fallback demo
# This path runs when ipywidgets is not available (e.g., plain Python execution).

if not 'UI_AVAILABLE' in globals() or not UI_AVAILABLE:
    print("Running fallback demo...")
    spec = Spec(
        "DemoSpec",
        "0.3",
        ["x", "y", "cap"],
        [
            {"type": "bounds", "var": "x", "lower": 0, "upper": 10},
            {"type": "equality", "expr": "Eq(x+y,cap)", "solve_for": "y"},
        ],
        [],
        {},
    )
    inputs = {"x": 12, "y": 3, "cap": 14}

    # Try the default strategy; you can also try strategy='altproj' for iterative repair.
    result = diagnose_and_repair(
        spec,
        inputs,
        strategy="bounds_then_equality",
        spec_path="(demo)",
        inputs_path="(demo)",
        verbose=True,
    )
    att = attest(spec.__dict__, result, spec_path="(demo)", inputs_path="(demo)")
    md = render_markdown("(demo)", "(demo)", result, att, verbose=True)
    ok = verify(att)

    print(badge(ok, "attestation verification"))
    print(md[:1200], "...")


## Notes for reviewers (why things are the way they are)

- **Offline-first**: outbound networking is blocked unless you explicitly opt in via `EQPROOF_ALLOW_NET=1`.
- **Non-goal**: this notebook is not trying to be a full unit algebra system; the unit model is intentionally small.
- **Repair strategies**:
  - **Bounds then equality**: best for many simple cases (fast, deterministic).
  - **Equality only**: useful when bounds are informational rather than enforced.
  - **Alternating projections**: helpful when equalities and bounds fight each other.

If you adapt this notebook to new domains, prefer adding new **constraints** and **examples** before adding complexity.
